# Eval Harness — Scoring the Student's Hop-1 Decompositions

Notebook 02 produced the labeled data and the 80/20 split. Here I score how well the fine-tuned student holds to the Hop-1 schema and gets the first lookup right, on the 95 held-out test questions.

The harness has two halves. **Scoring** is local and CPU-only, so I build it here. **Generation** needs the trained student, so it runs on Colab later. I build and check the scoring half first by feeding the gold teacher labels through it: they should score near-100%, which tells me the wiring is right before any real student output flows in.


In [1]:
import os
import getpass
from openai import OpenAI

# Prompt for the key once; it lives only in memory for this session, never in the notebook.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()
print("[INFO] OpenAI client ready")

[INFO] OpenAI client ready


## 1. Load the held-out test set

In [2]:
import json
from pathlib import Path
from collections import Counter

PREDS_FILE = Path("../data/preds_sys.jsonl")

def load_test(path: Path) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f]

test_rows = load_test(PREDS_FILE)

print(f"loaded {len(test_rows)} rows from {PREDS_FILE.name}")
print("type mix:", dict(Counter(r["type"] for r in test_rows)))
print("fields  :", list(test_rows[0].keys()))

loaded 95 rows from preds_sys.jsonl
type mix: {'comparison': 20, 'bridge': 75}
fields  : ['id', 'question', 'type', 'level', 'gold_titles', 'thought', 'action', 'target_entity', 'judge_reason', 'messages', 'pred_yaml']


In [3]:
import yaml

EXPECTED_FIELDS = {"thought", "action", "target_entity"}

def parse_and_validate(raw: str) -> dict:
    """Parse a teacher YAML string and validate the Hop-1 schema."""
    text = raw.strip()
    # Strip a stray ```yaml fence if the model wraps the output.
    if text.startswith("```"):
        text = text.strip("`")
        text = text[text.find("\n") + 1 :] if "\n" in text else text

    try:
        obj = yaml.safe_load(text)
    except yaml.YAMLError as e:
        raise ValueError(f"Not valid YAML: {e}\n---raw---\n{raw}")

    if not isinstance(obj, dict):
        raise ValueError(f"YAML did not parse to a mapping, got {type(obj)}:\n{raw}")
    if set(obj.keys()) != EXPECTED_FIELDS:
        raise ValueError(f"Field mismatch. Expected {EXPECTED_FIELDS}, got {set(obj.keys())}")
    if obj["action"] != "Lookup":
        raise ValueError(f"action must be 'Lookup', got {obj['action']!r}")

    return obj

In [4]:
JUDGE_PROMPT = """You are grading the FIRST step of a multi-hop question decomposition.

You are given:
- the QUESTION,
- a PROPOSED FIRST LOOKUP (the entity a model chose to look up first),
- the GOLD TITLES: the titles of the reference articles that hold the supporting facts for the question.

Decide one thing: would looking up the PROPOSED FIRST LOOKUP be a correct first step, one that leads to the gold supporting facts?

Count it as correct when the proposed lookup is the right starting entity, including when:
- it is the same real-world entity as a gold title under a different name,
- it is a description that denotes a gold entity,
- it is the right anchor even though the relevant fact is filed under a related gold title, 
- the question compares two named entities and the lookup is either one of them, since looking up either is a valid first step and which one
turns out to be the answer does not matter, or
- the question names no entity and the lookup is a faithful description of its subject (resolving that description is a later step, not part of this decision).

Count it as incorrect when the proposed lookup is only loosely related, is the unknown the question is trying to find, or is the wrong entity.

Reason briefly first, then give your verdict.
Return ONLY a JSON object: {"reason": "<one sentence>", "match": <true or false>}"""

In [5]:
import json

def grade_target_entity_llm(question: str, target_entity: str, gold_titles: list) -> dict:
    """LLM-judge: is target_entity a correct Hop-1 lookup for this question,
    given the gold supporting-fact titles? Return {'reason': str, 'match': bool}."""
    user = (
        f"Question: {question}\n"
        f"PROPOSED FIRST LOOKUP: {target_entity}\n"
        f"GOLD TITLES: {gold_titles}"
    )
    resp = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": user},
        ],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

In [6]:
def exact_match_to_teacher(pred_target, teacher_target) -> bool:
    return pred_target.strip().lower() == teacher_target.strip().lower()

In [7]:
def score_one(pred_yaml: str, gold_record: dict) -> dict:
    """Grades one student output, against its gold test row. Same keys on every path."""
    # The gate: format must parse and validate before anything downstream can be scored.
    try:
        pred = parse_and_validate(pred_yaml)
    except ValueError as e:
        return {"format_ok": False, "target_correct": False,
                "exact_match": False, "thought_ok": False, "error": str(e)}

    student_target = pred['target_entity'].strip().lower()
    if any(student_target == title.strip().lower() for title in gold_record['gold_titles']):
        target_correct = True
    else:
        verdict = grade_target_entity_llm(
            gold_record["question"], pred["target_entity"], gold_record['gold_titles']
        )
        target_correct = verdict['match']
    return {
        "format_ok": True,
        "target_correct": target_correct,
        "exact_match": exact_match_to_teacher(pred['target_entity'], gold_record['target_entity']),
        "thought_ok": pred['thought'].strip() != "",
            "error": None,
        }

## 2. Run the harness across the test set

In [8]:
import pandas as pd

records = []
for row in test_rows:
    pred_yaml = row["pred_yaml"]
    scores = score_one(pred_yaml, row)
    records.append({"type": row["type"], **scores})

scored = pd.DataFrame(records)
print(f"scored {len(scored)} rows")

scored 95 rows


In [9]:
METRICS = ["format_ok", "target_correct", "exact_match", "thought_ok"]

print(f"n = {len(scored)}   type mix = {scored['type'].value_counts().to_dict()}\n")
print("overall rates:")
print(scored[METRICS].mean().to_string())
print("\nby type:")
print(scored.groupby("type")[METRICS].mean().to_string())

# On the gold labels this should be empty; on real student output it holds the malformed rows.
bad = scored[~scored["format_ok"]]
print(f"\nnon-adherent rows: {len(bad)}")
if len(bad):
    print(bad[["type", "error"]].to_string())

n = 95   type mix = {'bridge': 75, 'comparison': 20}

overall rates:
format_ok         1.000000
target_correct    0.894737
exact_match       0.663158
thought_ok        1.000000

by type:
            format_ok  target_correct  exact_match  thought_ok
type                                                          
bridge            1.0        0.866667          0.6         1.0
comparison        1.0        1.000000          0.9         1.0

non-adherent rows: 0


## 3. Phase 1 benchmark — score all four models

Same `score_one` from above, run once per model's saved predictions, so the four numbers in the results log come from one shared scoring path, not four separate ones.

In [10]:
MODEL_FILES = {
    "untrained": "../data/preds_untrained_bench.jsonl",
    "untrained_teacherprompt": "../data/preds_untrained_teacherprompt_bench.jsonl",
    "sys": "../data/preds_sys_bench.jsonl",
    "groq_llama31_8b": "../data/preds_groq_bench.jsonl",
    "gpt4o_teacher": "../data/preds_gpt4o_bench.jsonl"

}

all_records = []
for model_name, path in MODEL_FILES.items():
    rows = load_test(Path(path))
    for row in rows:
        scores = score_one(row["pred_yaml"], row)
        all_records.append({
            "model": model_name,
            "type": row["type"],
            "latency_s": row["latency_s"],
            **scores,
        })

comparison = pd.DataFrame(all_records)
print(f"scored {len(comparison)} rows across {len(MODEL_FILES)} models")

scored 475 rows across 5 models


In [11]:
# Booleans average to a proportion here, so .mean() on format_ok/target_correct/
# exact_match/thought_ok directly gives the accuracy rate per model.
print("=== accuracy by model ===")
print(comparison.groupby("model")[METRICS].mean().to_string())

# median alongside mean for latency, since one slow API retry can drag the mean
# without reflecting the typical call.
print("\n=== latency by model (seconds) ===")
print(comparison.groupby("model")["latency_s"].agg(["mean", "median"]).to_string())

=== accuracy by model ===
                         format_ok  target_correct  exact_match  thought_ok
model                                                                      
gpt4o_teacher             1.000000        1.000000     0.936842    1.000000
groq_llama31_8b           1.000000        0.936842     0.726316    1.000000
sys                       0.989474        0.852632     0.621053    0.989474
untrained                 0.000000        0.000000     0.000000    0.000000
untrained_teacherprompt   0.210526        0.126316     0.084211    0.210526

=== latency by model (seconds) ===
                             mean    median
model                                      
gpt4o_teacher            1.887062  1.160474
groq_llama31_8b          5.428634  5.390845
sys                      4.810737  4.901618
untrained                1.624047  1.267144
untrained_teacherprompt  0.380625  0.078660


In [15]:
# Look at the actual successful rows.
rows = [json.loads(line) for line in open("../data/preds_untrained_teacherprompt_bench.jsonl")]

scored_rows = [{**row, **score_one(row['pred_yaml'], row)} for row in rows]

successes = [r for r in scored_rows if r['format_ok']]
print(f"{len(successes)} of {len(scored_rows)} had format_ok")

for r in successes[:5]:
    print(r['question'])
    print(repr(r['pred_yaml']))
    print()

20 of 95 had format_ok
What is the length of the River which has Wild Horse Creek as a tributary ?
'```yaml\nthought: "I need to find the river that has Wild Horse Creek as a tributary first to eventually identify the most populous country in Africa."\naction: "Lookup"\ntarget_entity: "Wild Horse Creek"\n```'

The Oberoi family is part of a hotel company that has a head office in what city?
'```yaml\nthought: "I need to find which hotel company the Oberoi family belongs to first."\naction: "Lookup"\ntarget_entity: "Oberoi family"\n```'

Are both Simon Wincer and Patrice Leconte film directors?
'```yaml\nthought: "I need to find which film director the Simon Wincer is."\naction: "Lookup"\ntarget_entity: "Simon Wincer"\n```'

Tura Beach, New South Wales is a suburb of a town that had what population at the 2016 census?
'thought: "I need to find the population of the town that had the most population at the 2016 census."\naction: "Lookup"\ntarget_entity: "the most populous country in Afri

In [18]:
example_questions = [
    "The Oberoi family is part of a hotel company that has a head office in what city?",
    "Were Scott Derrickson and Ed Wood of the same nationality?",
    "The wife of Arthur Miller starred in what movie?",
    "Cadmium Chloride is slightly soluble in this chemical, it is also called what?",
    "What language is most widely spoken in the most populous country in Africa?",
]

test_questions = {r['question'] for r in rows}
for q in example_questions:
    print(q in test_questions, "-", q)

True - The Oberoi family is part of a hotel company that has a head office in what city?
False - Were Scott Derrickson and Ed Wood of the same nationality?
False - The wife of Arthur Miller starred in what movie?
False - Cadmium Chloride is slightly soluble in this chemical, it is also called what?
False - What language is most widely spoken in the most populous country in Africa?
